<a href="https://colab.research.google.com/github/ohjisu320/AiTime/blob/master/%EA%B3%B5%EB%B6%80%EC%8B%9C%EA%B0%84_%EC%98%88%EC%B8%A1_%E2%80%94_Keras%EB%A1%9C_%EC%A7%81%EC%A0%91_%EB%A7%8C%EB%93%A4%EA%B3%A0_sklearn%EA%B3%BC_%EB%B9%84%EA%B5%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Keras로 같은 선형회귀 만들기 — sklearn과 비교 (공부시간 → 시험점수)
import numpy as np
from sklearn.linear_model import LinearRegression
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

tf.keras.utils.set_random_seed(42)                       # 난수 고정 → 실행마다 같은 결과

x = np.array([1, 3, 5, 8, 10]).reshape(5, 1)             # 공부시간 [[1],[3],[5],[8],[10]] → 모양 (5, 1) = 5명 × Feature 1개
y = np.array([10, 25, 60, 85, 100]).reshape(5, 1)        # 시험점수 → 모양 (5, 1)

# ---------- 1. sklearn: 공식으로 한 번에 ----------
sk = LinearRegression().fit(x, y)                        # 최소제곱법으로 W, b를 한 번에 계산
print("===== 1. sklearn =====")
print(f"W={sk.coef_[0][0]:.3f}  b={sk.intercept_[0]:.3f}  → 7시간: {sk.predict([[7]])[0][0]:.2f}점")

# ---------- 2. Keras: 경사하강법으로 Epoch 반복 ----------
model = Sequential()                                     # ① 모델 상자
model.add(Input(shape=(1,)))                             # ② 입력층: Feature 1개
model.add(Dense(units=1, activation="linear"))           #    출력층: ŷ = Wx + b
model.compile(optimizer=Adam(learning_rate=1e-2),        # ③ 옵티마이저(경사하강법 계열)
              loss="mse")                                #    손실함수 MSE
hist = model.fit(x, y, epochs=1000, verbose=0)           # ④ 학습: 예측 → 손실 → 역전파 → 수정을 1000번 반복 (verbose=0: 진행 출력 끔)

print("\n===== 2. Keras 학습 중 loss =====")
for e in [1, 10, 100, 500, 1000]:
    print(f"epoch {e:4d}: loss={hist.history['loss'][e - 1]:9.2f}")

W, b = model.get_layer(index=0).get_weights()            # 학습된 W, b 꺼내기 (W 모양 (1,1) → W[0][0], b 모양 (1,) → b[0])
pred7 = model.predict(np.array([[7]]), verbose=0)        # ⑤ 예측: 입력도 2차원 [[7]] → 결과도 2차원 [[62.41]] → pred7[0][0]
print("\n===== 3. Keras 결과 (1000 epoch) =====")
print(f"W={W[0][0]:.3f}  b={b[0]:.3f}  → 7시간: {pred7[0][0]:.2f}점")

# ---------- 3. 더 오래 학습하면? ----------
hist2 = model.fit(x, y, epochs=4000, verbose=0)          # 같은 모델로 이어서 4000번 더 (총 5000번)
W, b = model.get_layer(index=0).get_weights()
pred7 = model.predict(np.array([[7]]), verbose=0)
print("\n===== 4. Keras 결과 (총 5000 epoch) =====")
print(f"loss={hist2.history['loss'][-1]:.2f}  W={W[0][0]:.3f}  b={b[0]:.3f}  → 7시간: {pred7[0][0]:.2f}점")

===== 1. sklearn =====
W=10.395  b=-0.132  → 7시간: 72.63점

===== 2. Keras 학습 중 loss =====
epoch    1: loss=  3346.37
epoch   10: loss=  3272.49
epoch  100: loss=  2594.91
epoch  500: loss=   779.90
epoch 1000: loss=   127.03

===== 3. Keras 결과 (1000 epoch) =====
W=7.987  b=6.500  → 7시간: 62.41점
